# NLP Sentiment & Emotion Classification — Improved Recipe (Colab-ready)

Targets **≥ 94% test accuracy** on `dair-ai/emotion` (vs. 93% baseline).

| Lever | Original | Improved |
|---|---|---|
| Backbone | DistilBERT-base | **RoBERTa-base** |
| Pooling | mean only | **[CLS] ⊕ masked-mean concat** |
| Head | Dense(256) ReLU | **Dense(512) GELU + LayerNorm** |
| Output | softmax | **logits + `from_logits=True`** |
| Loss | sparse CE | **CE + label smoothing 0.05** |
| Optimizer | Adam 2e-5 | **AdamW (wd=0.01) + grad clip 1.0** |
| LR schedule | constant | **linear warmup (10%) → linear decay** |
| Max length | 128 | **96** |
| Batch size | 16 | **32** |
| Best-model metric | val_accuracy | **val_macro_f1** |

> Runtime → Change runtime type → **GPU** (T4 is fine). Then *Run all*.


## 1 — Install dependencies (no TF reinstall, so no runtime restart)

In [ ]:
# Use Colab's preinstalled TensorFlow as-is. Only add what's missing/old.
!pip install -q -U "transformers>=4.40,<4.50" "datasets>=2.16" "huggingface_hub>=0.23" \
                    scikit-learn matplotlib

## 2 — Imports, GPU detection, repro seed

In [ ]:
import os, sys, math, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, TFAutoModel
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score)

print("TF :", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'n/a')
print("GPU:", tf.config.list_physical_devices('GPU') or "⚠️  CPU only — training will be slow")

# Mixed precision is disabled by default for stability across Keras versions.
# Flip to True if you want the ~2x GPU speedup (requires recent Keras/TF).
USE_MIXED_PRECISION = False
if USE_MIXED_PRECISION and tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: ENABLED")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 3 — Model + tokenizer (inlined so the notebook is self-contained)

In [ ]:
EMOTION_LABELS = ["Sadness", "Joy", "Love", "Anger", "Fear", "Surprise"]
DEFAULT_MAX_LENGTH = 96


def load_tokenizer(model_name="roberta-base"):
    return AutoTokenizer.from_pretrained(model_name)


def tokenize(tokenizer, texts, max_length=DEFAULT_MAX_LENGTH, return_tensors="tf"):
    if not isinstance(texts, list):
        texts = list(texts)
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors=return_tensors,
    )


def build_backbone(model_name="roberta-base"):
    # Let transformers pick the best available weights format (TF/PT/safetensors).
    return TFAutoModel.from_pretrained(model_name, return_dict=False)


class EmotionClassifier(tf.keras.Model):
    """Transformer backbone + [CLS] ⊕ mean-pool head, logits output.

    Final dense layer is forced to float32 so the loss is numerically stable
    when running under `mixed_float16` global policy.
    """

    def __init__(self, backbone, num_classes=6, hidden_size=512,
                 dropout_pool=0.3, dropout_hidden=0.2, return_logits=True):
        super().__init__()
        self.backbone = backbone
        self.return_logits = return_logits
        self.dropout_pool = tf.keras.layers.Dropout(dropout_pool, name="dropout_pool")
        self.layer_norm = tf.keras.layers.LayerNormalization(name="pool_layer_norm")
        self.dense = tf.keras.layers.Dense(hidden_size, activation="gelu", name="hidden_proj")
        self.dropout_hidden = tf.keras.layers.Dropout(dropout_hidden, name="dropout_hidden")
        # dtype='float32' keeps logits in fp32 for stable softmax/loss under MP.
        self.classifier = tf.keras.layers.Dense(num_classes, name="logits", dtype="float32")

    def call(self, inputs, training=False):
        out = self.backbone(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            training=training,
        )
        token_embeddings = out[0] if isinstance(out, (tuple, list)) else out.last_hidden_state

        # Cast mask to match embedding dtype (works under fp32 and fp16)
        mask = tf.cast(inputs["attention_mask"], token_embeddings.dtype)
        mask = tf.expand_dims(mask, axis=-1)

        cls_embedding = token_embeddings[:, 0, :]
        masked = token_embeddings * mask
        sum_emb = tf.reduce_sum(masked, axis=1)
        sum_mask = tf.reduce_sum(mask, axis=1)
        mean_pool = sum_emb / tf.maximum(sum_mask, tf.cast(1e-9, token_embeddings.dtype))
        pooled = tf.concat([cls_embedding, mean_pool], axis=-1)

        x = self.dropout_pool(pooled, training=training)
        x = self.layer_norm(x)
        x = self.dense(x)
        x = self.dropout_hidden(x, training=training)
        logits = self.classifier(x)
        return logits if self.return_logits else tf.nn.softmax(logits, axis=-1)


print("Model + tokenizer helpers ready.")

## 4 — Hyperparameters

In [ ]:
MODEL_NAME      = "roberta-base"
MAX_LENGTH      = DEFAULT_MAX_LENGTH      # 96
BATCH_SIZE      = 32
EPOCHS          = 8
LR_PEAK         = 2e-5
WEIGHT_DECAY    = 0.01
WARMUP_FRAC     = 0.1
LABEL_SMOOTHING = 0.05
GRAD_CLIP_NORM  = 1.0
PATIENCE_ES     = 4

# Keras 3 requires the .weights.h5 suffix when saving weights only.
CKPT_DIR  = "model"
CKPT_PATH = os.path.join(CKPT_DIR, "best_emotion_model_roberta.weights.h5")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint will be saved to:", os.path.abspath(CKPT_PATH))

## 5 — Load `dair-ai/emotion`

In [ ]:
# trust_remote_code=True handles older snapshots of the dataset script
try:
    ds = load_dataset("dair-ai/emotion", trust_remote_code=True)
except TypeError:
    # Older `datasets` versions: kwarg doesn't exist yet
    ds = load_dataset("dair-ai/emotion")

print(ds)
lengths = [len(t.split()) for t in ds['train']['text']]
print(f"Word-length p50={np.percentile(lengths,50):.0f}  "
      f"p95={np.percentile(lengths,95):.0f}  "
      f"p99={np.percentile(lengths,99):.0f}  max={max(lengths)}")

## 6 — Tokenize all splits

In [ ]:
tokenizer = load_tokenizer(MODEL_NAME)

def encode_split(split):
    enc = tokenize(tokenizer, list(ds[split]['text']), max_length=MAX_LENGTH)
    labels = tf.constant(list(ds[split]['label']), dtype=tf.int32)
    return enc, labels

train_enc, y_train = encode_split('train')
val_enc,   y_val   = encode_split('validation')
test_enc,  y_test  = encode_split('test')

print("train:", y_train.shape, "val:", y_val.shape, "test:", y_test.shape)

In [ ]:
def make_ds(enc, labels, shuffle=False):
    d = tf.data.Dataset.from_tensor_slices((
        {"input_ids": enc['input_ids'], "attention_mask": enc['attention_mask']},
        labels,
    ))
    if shuffle:
        d = d.shuffle(10_000, seed=SEED)
    return d.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_enc, y_train, shuffle=True)
val_ds   = make_ds(val_enc,   y_val)
test_ds  = make_ds(test_enc,  y_test)

## 7 — Class weights (imbalance-aware)

In [ ]:
y_train_np = y_train.numpy()
classes = np.unique(y_train_np)
weights = compute_class_weight('balanced', classes=classes, y=y_train_np)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:")
for i, name in enumerate(EMOTION_LABELS):
    print(f"  {name:<10s} {class_weight[i]:.3f}")

## 8 — Build the model

In [ ]:
backbone = build_backbone(MODEL_NAME)
backbone.trainable = True
model = EmotionClassifier(backbone, num_classes=6, return_logits=True)

# Build by passing a dummy batch
dummy = tokenizer("init", return_tensors="tf", padding="max_length", max_length=8)
_ = model({"input_ids": dummy["input_ids"], "attention_mask": dummy["attention_mask"]})

model.summary()

## 9 — Optimizer (AdamW + warmup→linear decay) and label-smoothed loss

In [ ]:
steps_per_epoch = math.ceil(len(y_train) / BATCH_SIZE)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_FRAC))

class WarmupLinearDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        super().__init__()
        self.peak_lr      = float(peak_lr)
        self.warmup_steps = int(max(1, warmup_steps))
        self.total_steps  = int(max(self.warmup_steps + 1, total_steps))

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / float(self.warmup_steps))
        decay_lr  = self.peak_lr * tf.maximum(
            0.0,
            (float(self.total_steps) - step) /
            float(self.total_steps - self.warmup_steps),
        )
        return tf.where(step < float(self.warmup_steps), warmup_lr, decay_lr)

    def get_config(self):
        return dict(peak_lr=self.peak_lr,
                    warmup_steps=self.warmup_steps,
                    total_steps=self.total_steps)

lr_schedule = WarmupLinearDecay(LR_PEAK, warmup_steps, total_steps)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    global_clipnorm=GRAD_CLIP_NORM,
)

class SparseCEWithLabelSmoothing(tf.keras.losses.Loss):
    """Sparse interface + label smoothing (Keras CCE doesn't allow LS on sparse)."""
    def __init__(self, num_classes, label_smoothing=0.05, **kw):
        super().__init__(**kw)
        self.cce = tf.keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=label_smoothing,
        )
        self.num_classes = num_classes
    def call(self, y_true, y_pred):
        y_true = tf.one_hot(tf.cast(y_true, tf.int32), self.num_classes)
        y_pred = tf.cast(y_pred, tf.float32)  # safe under mixed precision
        return self.cce(y_true, y_pred)

loss_fn = SparseCEWithLabelSmoothing(6, label_smoothing=LABEL_SMOOTHING)
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
print(f"Optimizer + loss ready. warmup_steps={warmup_steps} total_steps={total_steps}")

## 10 — Callbacks: track val_macro_f1 and checkpoint on it

In [ ]:
class MacroF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, val_labels):
        super().__init__()
        self.val_ds     = val_ds
        self.val_labels = np.asarray(val_labels)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logits = self.model.predict(self.val_ds, verbose=0)
        preds  = np.argmax(logits, axis=1)
        f1 = f1_score(self.val_labels, preds, average='macro')
        logs['val_macro_f1'] = f1
        print(f"  val_macro_f1: {f1:.4f}")

callbacks = [
    MacroF1Callback(val_ds, y_val.numpy()),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max',
        patience=PATIENCE_ES, restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_PATH, monitor='val_macro_f1', mode='max',
        save_best_only=True, save_weights_only=True, verbose=1,
    ),
]

## 11 — Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print("\nTraining done. Best weights restored.")

## 12 — Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'],     label='train acc')
ax[0].plot(history.history['val_accuracy'], label='val acc')
ax[0].set_title('Accuracy'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history['loss'],     label='train loss')
ax[1].plot(history.history['val_loss'], label='val loss')
ax[1].set_title('Loss'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 13 — Test-set evaluation

In [ ]:
test_logits = model.predict(test_ds, verbose=1)
test_probs  = tf.nn.softmax(test_logits, axis=-1).numpy()
test_preds  = np.argmax(test_probs, axis=1)

acc  = accuracy_score(y_test.numpy(), test_preds)
mf1  = f1_score(y_test.numpy(), test_preds, average='macro')
wf1  = f1_score(y_test.numpy(), test_preds, average='weighted')

print("="*60)
print(f"Test accuracy : {acc*100:.2f}%")
print(f"Macro F1      : {mf1:.4f}")
print(f"Weighted F1   : {wf1:.4f}")
print("="*60)
print(classification_report(y_test.numpy(), test_preds, target_names=EMOTION_LABELS))

In [ ]:
cm = confusion_matrix(y_test.numpy(), test_preds)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap='Blues')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xticks(range(6), EMOTION_LABELS, rotation=45)
plt.yticks(range(6), EMOTION_LABELS)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

## 14 — Save & download weights

The best weights are already at `CKPT_PATH` (saved by `ModelCheckpoint`).
We also re-save explicitly so you can grab them after training.

In [ ]:
model.save_weights(CKPT_PATH)
size_mb = os.path.getsize(CKPT_PATH) / 1024**2
print(f"Saved → {CKPT_PATH}  ({size_mb:.1f} MB)")

### (Optional) Download to your computer

After training, drop the weights into your repo's `model/` directory and
relaunch the Streamlit app — it auto-detects this file.

In [ ]:
# In Colab this triggers a browser download; safe to skip otherwise.
try:
    from google.colab import files
    files.download(CKPT_PATH)
except Exception as e:
    print("Not running in Colab or download skipped:", e)

### (Optional) Mount Google Drive and copy weights there

In [ ]:
# Uncomment to persist weights to your Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(CKPT_PATH, '/content/drive/MyDrive/best_emotion_model_roberta.weights.h5')
# print("Copied to Drive.")